In [0]:
# ============================================
# BANKING SYNAPSE → DATABRICKS MIGRATION
# Framework Configuration
# ============================================

CATALOG = "databricks_wrkspce"
SCHEMA = "default"
VOLUME = "banking_data"

# Source data
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/raw"

# New migration output paths
MIGRATION_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/migration"

BRONZE_PATH = f"{MIGRATION_PATH}/bronze"
SILVER_PATH = f"{MIGRATION_PATH}/silver"
GOLD_PATH = f"{MIGRATION_PATH}/gold"

# Framework configuration
CONFIG_PATH = f"{MIGRATION_PATH}/config"

print("Raw Path    :", RAW_PATH)
print("Bronze Path :", BRONZE_PATH)
print("Silver Path :", SILVER_PATH)
print("Gold Path   :", GOLD_PATH)
print("Config Path :", CONFIG_PATH)

Raw Path    : /Volumes/databricks_wrkspce/default/banking_data/raw
Bronze Path : /Volumes/databricks_wrkspce/default/banking_data/migration/bronze
Silver Path : /Volumes/databricks_wrkspce/default/banking_data/migration/silver
Gold Path   : /Volumes/databricks_wrkspce/default/banking_data/migration/gold
Config Path : /Volumes/databricks_wrkspce/default/banking_data/migration/config


In [0]:

display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/accounts.csv,accounts.csv,1173488,1787045648000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/branches.csv,branches.csv,9865,1787045642000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/customers.json,customers.json,2844672,1787045643000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/transactions.csv,transactions.csv,37966927,1787045656000


In [0]:
#Configurations

SOURCE_CONFIG = {
    "accounts": {
        "file": f"{RAW_PATH}/accounts.csv",
        "format": "csv"
    },

    "branches": {
        "file": f"{RAW_PATH}/branches.csv",
        "format": "csv"
    },

    "customers": {
        "file": f"{RAW_PATH}/customers.json",
        "format": "json"
    },

    "transactions": {
        "file": f"{RAW_PATH}/transactions.csv",
        "format": "csv"
    }
}

for name, config in SOURCE_CONFIG.items():
    print(f"{name:15} | {config['format']:5} | {config['file']}")

accounts        | csv   | /Volumes/databricks_wrkspce/default/banking_data/raw/accounts.csv
branches        | csv   | /Volumes/databricks_wrkspce/default/banking_data/raw/branches.csv
customers       | json  | /Volumes/databricks_wrkspce/default/banking_data/raw/customers.json
transactions    | csv   | /Volumes/databricks_wrkspce/default/banking_data/raw/transactions.csv


In [0]:
# ============================================
# STEP 4 — GENERIC INGESTION
# ============================================

def read_source(config):
    """
    Reads a source file based on its configured format.
    Supports CSV and JSON.
    """

    file_path = config["file"]
    file_format = config["format"]

    if file_format == "csv":
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(file_path)
        )

    elif file_format == "json":
        df = spark.read.option("multiLine", True).json(file_path)

    else:
        raise ValueError(
            f"Unsupported source format: {file_format}"
        )

    return df

In [0]:
# ============================================
# STEP 5 — READ ALL SOURCES
# ============================================

dataframes = {}

for dataset_name, config in SOURCE_CONFIG.items():

    print(f"Reading {dataset_name}...")

    df = read_source(config)

    dataframes[dataset_name] = df

    print(f"  Records: {df.count()}")
    print(f"  Columns: {len(df.columns)}")
    print()

Reading accounts...
  Records: 20050
  Columns: 7

Reading branches...
  Records: 200
  Columns: 5

Reading customers...
  Records: 10100
  Columns: 9

Reading transactions...
  Records: 501000
  Columns: 8



In [0]:
raw_path = "/Volumes/databricks_wrkspce/default/banking_data/raw"

display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/accounts.csv,accounts.csv,1173488,1787045648000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/branches.csv,branches.csv,9865,1787045642000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/customers.json,customers.json,2844672,1787045643000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/raw/transactions.csv,transactions.csv,37966927,1787045656000


In [0]:
# ============================================
# GENERIC BRONZE WRITER
# ============================================

def write_bronze(df, dataset_name):
    """
    Writes a source DataFrame to the migration Bronze layer
    as Delta.
    """

    target_path = f"{BRONZE_PATH}/{dataset_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )

    print(f"{dataset_name} → Bronze written successfully")

In [0]:
for dataset_name, df in dataframes.items():

    write_bronze(
        df,
        dataset_name
    )

accounts → Bronze written successfully
branches → Bronze written successfully
customers → Bronze written successfully
transactions → Bronze written successfully


In [0]:
accounts_bronze = spark.read.format("delta").load(
    f"{BRONZE_PATH}/accounts"
)

display(accounts_bronze)

account_id,customer_id,account_type,account_open_date,account_status,balance,branch_id
A0000001,C002495,Salary,2025-11-25,Closed,469741.3,B0127
A0000002,C000503,Savings,2026-05-15,Active,148270.59,B0073
A0000003,C009352,Salary,2023-01-19,Active,86254.91,B0067
A0000004,C001600,Savings,2022-04-19,Active,271196.87,B0168
A0000005,C008418,Savings,2021-12-24,Active,408274.3,B0128
A0000006,C008027,Salary,2021-09-24,Closed,11829.57,B0031
A0000007,C008392,Savings,2021-09-02,Active,383199.1,B0060
A0000008,C007713,Salary,2022-07-21,Active,399871.0,B0102
A0000009,C004165,Current,2023-10-27,Active,171690.94,B0053
A0000010,C001073,Salary,2026-05-25,Dormant,67339.37,B0118
